In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler , OneHotEncoder , TargetEncoder  , PowerTransformer
from sklearn.model_selection import train_test_split 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.base import BaseEstimator , TransformerMixin

In [2]:
pd.set_option('display.max_columns' , None)

In [3]:
df = pd.read_csv('../data/processed/processed_NYC.csv')

In [4]:
df.sample(5)

,vendor_id,passenger_count,store_and_fwd_flag,pickup_zone,pickup_state,dropoff_zone,dropoff_state,distance_haversine_km,distance_manhattan_km,direction,hour_of_day,day_of_week,month_of_year,is_rush_hour,season_name,trip_duration_min
833276,2,6,0,Long Island City,New York,New York City,New York,2.191508,2.977968,-118.919337,0,6,2,0,Winter,12.70
1404809,1,2,0,New York City,New York,New York City,New York,3.151818,3.186984,-0.642610,3,5,2,0,Winter,14.05
347300,2,1,0,The Bronx,New York,Manhattan,New York,7.575390,8.263638,-95.449100,21,6,2,0,Winter,19.88
304204,1,1,1,Manhattan,New York,Weehawken,New Jersey,3.259363,4.491147,-121.998936,10,0,5,1,Spring,26.50
479428,2,1,0,Manhattan,New York,Long Island City,New York,2.685076,3.598721,-153.596226,12,2,6,0,Summer,16.02


In [5]:
X = df.drop(columns="trip_duration_min")
y = df['trip_duration_min']

In [6]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [7]:
class NYCTaxiFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Custom transformer that extracts advanced spatial vectors,
    spatio-temporal intersections, and temporal window flags.
    """
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # 1. Advanced Spatio-Temporal Ensembles
        route_comb = X['pickup_zone'].astype(str) + ' -> ' + X['dropoff_zone'].astype(str)
        time_bin = pd.cut(
            X['hour_of_day'], 
            bins=[0, 6, 12, 16, 20, 24], 
            labels=['Night', 'Morning', 'Midday', 'Evening_Rush', 'Late_Night'], 
            include_lowest=True
        ).astype(str)
        
        X['route_time_density'] = route_comb + " @ " + time_bin

        # 2. Advanced Spatial Transformations
        X['is_interstate_trip'] = (X['pickup_state'].astype(str) != X['dropoff_state'].astype(str)).astype(int)

        X['travel_quadrant'] = pd.cut(
            X['direction'], 
            bins=[-180, -90, 0, 90, 180], 
            labels=['South-West', 'North-West', 'North-East', 'South-East'],
            include_lowest=True
        ).astype(str)

        # 3. Advanced Temporal Transformation Contexts
        X['is_late_night'] = X['hour_of_day'].between(0, 5).astype(int)
        X['is_weekend_night'] = (X['is_late_night'] & (X['day_of_week'].isin([4, 5, 6]))).astype(int)

        # Explicitly remove temporary intermediate column states if we do not want them saved
        # X = X.drop(columns=['route_combination', 'time_bin'], errors='ignore')

        return X
    
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return None
        
        # Safe structural type check conversion
        input_features_list = list(input_features)
        
        # Structural fields added inside this transformer class
        new_features = [
            "route_time_density", 
            "is_interstate_trip", 
            "travel_quadrant", 
            "is_late_night", 
            "is_weekend_night"
        ]
        
        return np.array(input_features_list + new_features)

In [8]:
target_encoding_features = ["pickup_zone" , "dropoff_zone" , "route_time_density"]
ohe_features = ['dropoff_state' , 'pickup_state' , 'travel_quadrant' , 'season_name']

numerical_col = [
    'vendor_id', 'passenger_count', 'store_and_fwd_flag',
    'distance_haversine_km', 'distance_manhattan_km', 'direction',
    'hour_of_day', 'day_of_week', 'month_of_year', 'is_rush_hour',
    'is_interstate_trip', 'is_late_night', 'is_weekend_night'
]

In [9]:
column_transformer = ColumnTransformer(
    transformers=[
        ("target_encoded" , TargetEncoder(smooth="auto") , target_encoding_features),
        ("Ohe" , OneHotEncoder(handle_unknown="ignore") , ohe_features),
        ("std_scalar" , StandardScaler() , numerical_col)
    ],remainder="passthrough"
)

In [ ]:
pipeline = Pipeline(
    steps=[
        ("feature_engineering" , NYCTaxiFeatureEngineer()),
        ("column_transformer" , column_transformer)
    ]
)

In [11]:
pipeline.fit(X_train , y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('feature_engineering', ...), ('column_transformer', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('target_encoded', ...), ('Ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold

In [12]:
pipeline.get_feature_names_out()

array(['target_encoded__pickup_zone', 'target_encoded__dropoff_zone',
       'target_encoded__route_time_density',
       'Ohe__dropoff_state_Connecticut', 'Ohe__dropoff_state_New Jersey',
       'Ohe__dropoff_state_New York', 'Ohe__pickup_state_New Jersey',
       'Ohe__pickup_state_New York', 'Ohe__travel_quadrant_North-East',
       'Ohe__travel_quadrant_North-West',
       'Ohe__travel_quadrant_South-East',
       'Ohe__travel_quadrant_South-West', 'Ohe__season_name_Spring',
       'Ohe__season_name_Summer', 'Ohe__season_name_Winter',
       'std_scalar__vendor_id', 'std_scalar__passenger_count',
       'std_scalar__store_and_fwd_flag',
       'std_scalar__distance_haversine_km',
       'std_scalar__distance_manhattan_km', 'std_scalar__direction',
       'std_scalar__hour_of_day', 'std_scalar__day_of_week',
       'std_scalar__month_of_year', 'std_scalar__is_rush_hour',
       'std_scalar__is_interstate_trip', 'std_scalar__is_late_night',
       'std_scalar__is_weekend_night'], dty

In [13]:
train_trans = pipeline.fit_transform(X_train , y_train)

X_train_trans = pd.DataFrame(train_trans , columns=pipeline.get_feature_names_out())

In [14]:
test_trans = pipeline.transform(X_test)

X_test_trans = pd.DataFrame(test_trans , columns=pipeline.get_feature_names_out())

In [15]:
X_train_trans

,target_encoded__pickup_zone,target_encoded__dropoff_zone,target_encoded__route_time_density,Ohe__dropoff_state_Connecticut,Ohe__dropoff_state_New Jersey,Ohe__dropoff_state_New York,Ohe__pickup_state_New Jersey,Ohe__pickup_state_New York,Ohe__travel_quadrant_North-East,Ohe__travel_quadrant_North-West,Ohe__travel_quadrant_South-East,Ohe__travel_quadrant_South-West,Ohe__season_name_Spring,Ohe__season_name_Summer,Ohe__season_name_Winter,std_scalar__vendor_id,std_scalar__passenger_count,std_scalar__store_and_fwd_flag,std_scalar__distance_haversine_km,std_scalar__distance_manhattan_km,std_scalar__direction,std_scalar__hour_of_day,std_scalar__day_of_week,std_scalar__month_of_year,std_scalar__is_rush_hour,std_scalar__is_interstate_trip,std_scalar__is_late_night,std_scalar__is_weekend_night
0,12.626611,13.880436,14.062392,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-1.072595,-0.506400,-0.074345,-0.201545,-0.146881,-1.244257,-1.034514,-0.536874,-0.902287,1.521023,-0.504921,-0.363556,-0.288517
1,12.628391,13.556622,21.882787,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,-1.072595,-0.506400,13.450777,0.968920,1.014628,-1.308925,-0.565809,-1.560422,0.882214,1.521023,-0.504921,-0.363556,-0.288517
2,13.930868,13.243359,14.056330,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.932318,3.299085,-0.074345,-0.378048,-0.366702,0.356233,-0.722044,0.998447,-0.307454,-0.657452,1.980509,-0.363556,-0.288517
3,12.628391,12.481656,7.874424,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.932318,2.537988,-0.074345,-0.699119,-0.716808,-0.683642,1.152776,-0.025100,-0.307454,-0.657452,-0.504921,-0.363556,-0.288517
4,12.626611,13.531391,14.533779,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.932318,-0.506400,-0.074345,-0.255660,-0.191039,-1.164568,-1.971924,0.998447,0.882214,-0.657452,-0.504921,2.750610,3.466000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1125737,12.628547,12.468878,8.924742,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.932318,-0.506400,-0.074345,-0.554584,-0.524894,-0.109546,0.527836,1.510221,-0.902287,-0.657452,-0.504921,-0.363556,-0.288517
1125738,13.000265,13.911523,8.841273,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,-1.072595,-0.506400,-0.074345,-0.473624,-0.539877,0.129621,0.840306,0.486673,0.287380,1.521023,-0.504921,-0.363556,-0.288517
1125739,13.928436,12.481656,18.670829,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,-1.072595,1.776891,-0.074345,0.059920,0.010917,0.297517,1.309011,-0.025100,0.882214,-0.657452,-0.504921,-0.363556,-0.288517
1125740,13.930868,13.531391,9.874056,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,-1.072595,-0.506400,-0.074345,-0.748669,-0.720068,1.297760,-0.565809,-1.560422,-0.307454,1.521023,-0.504921,-0.363556,-0.288517
